# Chapter 6 · Communication Protocols Between Agents
**Mastering Agentic AI** — Manning Publications



## What this notebook covers
- Agent-to-Agent (A2A) protocols and message envelopes
- Human-to-Agent (H2A) and Agent-to-Human (A2H) structured handshakes
- Agent Communication Protocol (ACP) — capability registry and routing
- Four-agent Diet Coach system: Orchestrator, NutritionAnalyst, MealPlanner, BehaviourCoach
- Coordination, negotiation, and conflict resolution

---

## 6.0 · Setup

In [ ]:
# !pip install anthropic python-dotenv
import os, json, uuid, asyncio
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "See appendix_a_api_keys.md"
client = Anthropic()
MODEL = "claude-opus-4-5"
print("Ready")

## 6.1 · The Message Envelope (A2A Protocol)

In [ ]:
class MessageType(str, Enum):
    REQUEST  = "request"
    RESPONSE = "response"
    DELEGATE = "delegate"
    RESULT   = "result"
    ERROR    = "error"

@dataclass
class AgentMessage:
    """
    Standard message envelope for agent-to-agent communication.
    Inspired by FIPA ACL and the Google A2A draft spec.
    
    Why an envelope? Without one, agents pass raw strings.
    Routing, debugging, and tracing all become impossible at scale.
    """
    sender:     str
    recipient:  str
    msg_type:   MessageType
    content:    Any
    message_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    reply_to:   str | None = None
    metadata:   dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        d = asdict(self)
        d["msg_type"] = self.msg_type.value
        return d

class MessageBus:
    """
    In-process message bus. Production replacement: Redis pub/sub or RabbitMQ.
    """
    def __init__(self):
        self._queues: dict[str, list[AgentMessage]] = {}

    def register(self, agent_id: str) -> None:
        self._queues.setdefault(agent_id, [])

    def send(self, message: AgentMessage) -> None:
        self._queues.setdefault(message.recipient, []).append(message)

    def receive(self, agent_id: str) -> list[AgentMessage]:
        msgs = self._queues.get(agent_id, [])
        self._queues[agent_id] = []
        return msgs

# Test
bus = MessageBus()
bus.register("nutrition_analyst")
bus.send(AgentMessage(
    sender="orchestrator", recipient="nutrition_analyst",
    msg_type=MessageType.DELEGATE,
    content={"task": "analyse", "meal": "150g salmon, 200g broccoli"}
))
msgs = bus.receive("nutrition_analyst")
print(f"Received {len(msgs)} message(s):")
print(json.dumps(msgs[0].to_dict(), indent=2))

## 6.2 · H2A and A2H Structured Handshakes

In [ ]:
@dataclass
class H2ARequest:
    """Human-to-Agent request with explicit capability declaration."""
    user_id:             str
    message:             str
    capabilities_needed: list[str]  # e.g. ["nutrition_analysis", "meal_planning"]
    context:             dict = field(default_factory=dict)

@dataclass
class A2HResponse:
    """Agent-to-Human response with confidence and source attribution."""
    message:    str
    confidence: float      # 0.0 – 1.0
    sources:    list[str]  # which sub-agents contributed
    follow_up:  str | None = None

# The explicit capability declaration lets the orchestrator route without
# guessing. It also gives the user visibility into what the system will do.
req = H2ARequest(
    user_id="alex",
    message="I want a high-protein meal plan for tomorrow",
    capabilities_needed=["nutrition_analysis", "meal_planning"],
    context={"goal": "lose 5kg", "restriction": "lactose intolerant"}
)
print("H2A Request:", json.dumps(asdict(req), indent=2))

## 6.3 · Agent Registry (ACP — Capability Discovery)

In [ ]:
class AgentRegistry:
    """
    A capability registry so the orchestrator can discover agents at runtime.
    In production: replace with a service mesh or a dedicated discovery service.
    """
    def __init__(self):
        self._registry: dict[str, dict] = {}

    def register(self, agent_id: str, capabilities: list[str], description: str) -> None:
        self._registry[agent_id] = {
            "capabilities": capabilities,
            "description": description,
            "status": "available",
        }

    def find(self, capability: str) -> list[str]:
        """Return agent IDs that have the requested capability."""
        return [
            aid for aid, info in self._registry.items()
            if capability in info["capabilities"]
        ]

    def describe(self) -> str:
        lines = []
        for aid, info in self._registry.items():
            caps = ", ".join(info["capabilities"])
            lines.append(f"  {aid}: {caps}")
        return "\n".join(lines)

registry = AgentRegistry()
registry.register("nutrition_analyst", ["nutrition_analysis", "macro_calculation"],
                  "Analyses meals for macro and micronutrient content")
registry.register("meal_planner",      ["meal_planning", "recipe_suggestion"],
                  "Creates personalised daily meal plans")
registry.register("behaviour_coach",   ["habit_coaching", "motivation"],
                  "Supports habit formation and accountability")
print("Registry:")
print(registry.describe())
print()
print("Agents for 'meal_planning':", registry.find("meal_planning"))

## 6.4 · The Four-Agent Diet Coach System

In [ ]:
AGENT_CONFIGS = {
    "nutrition_analyst": {
        "role": "Nutrition Analyst",
        "system": (
            "You are a Nutrition Analyst specialising in macro and micronutrient analysis. "
            "Your ONLY job is to analyse nutritional content. "
            "Return structured JSON: {analysis: str, macros: {protein, carbs, fat, fibre}, "
            "deficits: [str], flags: [str]}. Do NOT give coaching advice."
        )
    },
    "meal_planner": {
        "role": "Meal Planner",
        "system": (
            "You are a Meal Planner. Given nutritional analysis and user constraints, "
            "suggest specific, practical meals. "
            "Return JSON: {meals: [{name, ingredients, calories, protein_g}], "
            "reasoning: str}. Keep suggestions realistic for a home cook."
        )
    },
    "behaviour_coach": {
        "role": "Behaviour Coach",
        "system": (
            "You are a Behaviour Coach specialising in dietary habit formation. "
            "Given a meal plan and user context, craft one encouraging, actionable message. "
            "Return JSON: {message: str, one_thing_to_do_today: str, confidence_score: float}"
        )
    },
}

def run_specialist(agent_id: str, task_content: dict) -> dict:
    """Run a specialist agent and parse its structured JSON response."""
    config = AGENT_CONFIGS[agent_id]
    prompt = json.dumps(task_content, indent=2)
    response = client.messages.create(
        model=MODEL, max_tokens=512, system=config["system"],
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    # Strip markdown fences if present
    import re
    raw = re.sub(r'^```(?:json)?\n?', '', raw).rstrip('`').strip()
    try:
        return json.loads(raw)
    except:
        return {"raw_output": raw}  # fallback

# Test NutritionAnalyst
analysis = run_specialist("nutrition_analyst", {
    "meal_description": "150g grilled salmon with 200g steamed broccoli and 100g brown rice",
    "user_goal": "lose 5kg, high protein"
})
print("Nutrition Analyst output:")
print(json.dumps(analysis, indent=2)[:600])

In [ ]:
class OrchestratorAgent:
    """
    Section 6.4: Orchestrator that routes requests to specialists and synthesises outputs.
    
    Design principle: the orchestrator does NOT do specialist work.
    It routes, coordinates, and synthesises. Specialists do the analysis.
    """
    def __init__(self, registry: AgentRegistry):
        self.registry = registry
        self.client   = Anthropic()

    def _route(self, request: H2ARequest) -> list[str]:
        """Determine which agents to call based on required capabilities."""
        agents = []
        for capability in request.capabilities_needed:
            found = self.registry.find(capability)
            agents.extend(found)
        return list(dict.fromkeys(agents))  # deduplicate, preserve order

    def handle(self, request: H2ARequest) -> A2HResponse:
        agents_to_call = self._route(request)
        results = {}

        # Call each specialist in sequence (Chapter 7 adds parallel execution)
        context = {"user_message": request.message, "user_context": request.context}
        for agent_id in agents_to_call:
            print(f"  → Calling {agent_id}...")
            result = run_specialist(agent_id, context)
            results[agent_id] = result
            context[f"{agent_id}_output"] = result  # chain outputs

        # Synthesise into final response
        synthesis_prompt = (
            f"User asked: {request.message}\n\n"
            f"Specialist outputs: {json.dumps(results, indent=2)}\n\n"
            "Write a concise, friendly 2-3 sentence response for the user."
        )
        synthesis = self.client.messages.create(
            model=MODEL, max_tokens=256,
            messages=[{"role": "user", "content": synthesis_prompt}]
        )
        final_msg = synthesis.content[0].text

        return A2HResponse(
            message=final_msg,
            confidence=0.85,
            sources=agents_to_call,
            follow_up="Would you like me to adjust the meal plan?"
        )

orch = OrchestratorAgent(registry)
req = H2ARequest(
    user_id="alex",
    message="Give me a high-protein dinner idea for tonight",
    capabilities_needed=["nutrition_analysis", "meal_planning"],
    context={"goal": "lose 5kg", "restriction": "lactose intolerant", "max_calories": 500}
)
response = orch.handle(req)
print("\nFinal response to user:")
print(response.message)
print(f"Sources: {response.sources}")

## 6.5 · Chapter Summary

| Concept | What we built |
|---|---|
| AgentMessage | Standard envelope: sender, recipient, type, content, message_id |
| MessageBus | In-process routing (upgrade path: Redis pub/sub) |
| H2ARequest / A2HResponse | Structured handshake with capability declaration and confidence |
| AgentRegistry | Capability discovery — orchestrator finds agents by capability, not name |
| OrchestratorAgent | Routes → calls specialists → chains outputs → synthesises |

**What is next?** Chapter 7 — Evaluation: measuring whether your agent system actually works.

---
*Mastering Agentic AI · Manning Publications*